In [1]:
# ============================================================
# STAGE 9C — FEATURE MAPPER VALIDATION
# ============================================================

import json
import sys
from pathlib import Path

import pandas as pd


# ============================================================
# 1. PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print("=" * 70)
print("STAGE 9C — FEATURE MAPPER VALIDATION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)


# ============================================================
# 2. BACKEND PATH
# ============================================================

BACKEND_DIR = (
    PROJECT_ROOT / "backend"
)

if not BACKEND_DIR.exists():
    raise FileNotFoundError(
        f"Backend directory not found:\n{BACKEND_DIR}"
    )

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BACKEND_DIR)
    )


# ============================================================
# 3. LOAD FEATURE MAPPER
# ============================================================

print("\n" + "=" * 70)
print("LOADING FEATURE MAPPER")
print("=" * 70)

from feature_mapper import (
    build_56_feature_vector
)

print("\n✓ feature_mapper imported successfully")


# ============================================================
# 4. LOAD PRODUCTION SCHEMA
# ============================================================

FEATURE_FILE = (
    PROJECT_ROOT
    / "models"
    / "final_model_features.json"
)

if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_FILE}"
    )

with open(
    FEATURE_FILE,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)


production_features = schema.get(
    "all_features",
    []
)

if not production_features:

    production_features = (
        schema.get(
            "categorical_features",
            []
        )
        +
        schema.get(
            "numeric_features",
            []
        )
    )


print("\nProduction schema loaded")

print(
    "Expected features:",
    len(production_features)
)


# ============================================================
# 5. CREATE REALISTIC TEST INPUT
# ============================================================

TEST_IMAGE = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "instagram_data"
    / "img"
    / "insta1.jpg"
)


print("\n" + "=" * 70)
print("TEST INPUT")
print("=" * 70)

print("\nCaption:")

test_caption = (
    "Amazing sunset in Sri Lanka! 🌅✨ "
    "Such a beautiful evening by the beach."
)

print(test_caption)

print("\nHashtags:")

test_hashtags = (
    "#srilanka #travel #sunset "
    "#photography #beach"
)

print(test_hashtags)

print("\nImage:")

print(TEST_IMAGE)

print(
    "\nImage exists:",
    TEST_IMAGE.exists()
)


# ============================================================
# 6. BUILD 56-FEATURE VECTOR
# ============================================================

print("\n" + "=" * 70)
print("BUILDING PRODUCTION FEATURE VECTOR")
print("=" * 70)


feature_vector = build_56_feature_vector(

    caption=test_caption,

    hashtags=test_hashtags,

    category="Travel",

    account_type="Creator",

    follower_count=125000,

    following_count=850,

    account_age_days=1450,

    verified_status=0,

    posting_frequency=4.0,

    average_historical_engagement=0.052,

    audience_growth_rate=0.018,

    account_activity_level=0.75,

    content_consistency=0.70,

    posting_hour=18,

    day_of_week="Saturday",

    posting_time_period="Evening",

    media_type="Image",

    has_location=1,

    sponsored=0,

    content_originality=0.75,

    content_quality_score=0.80,

    creator_activity_score=0.72,

    image_path=str(TEST_IMAGE)
)


print("\n✓ Feature vector generated")


# ============================================================
# 7. BASIC SHAPE
# ============================================================

print("\n" + "=" * 70)
print("VECTOR SHAPE VALIDATION")
print("=" * 70)

print(
    "\nRows:",
    feature_vector.shape[0]
)

print(
    "Columns:",
    feature_vector.shape[1]
)


# ============================================================
# 8. FEATURE NAME COMPARISON
# ============================================================

generated_features = list(
    feature_vector.columns
)


missing_features = [
    feature
    for feature in production_features
    if feature not in generated_features
]


unexpected_features = [
    feature
    for feature in generated_features
    if feature not in production_features
]


print("\n" + "=" * 70)
print("FEATURE NAME VALIDATION")
print("=" * 70)


print(
    "\nMissing features:",
    len(missing_features)
)

if missing_features:

    for feature in missing_features:
        print(
            "  -",
            feature
        )


print(
    "\nUnexpected features:",
    len(unexpected_features)
)

if unexpected_features:

    for feature in unexpected_features:
        print(
            "  -",
            feature
        )


# ============================================================
# 9. ORDER VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FEATURE ORDER VALIDATION")
print("=" * 70)

order_matches = (
    generated_features
    ==
    production_features
)


print(
    "\nExact feature order:",
    order_matches
)


if not order_matches:

    print(
        "\nFirst feature mismatch:"
    )

    for index, (
        expected,
        generated
    ) in enumerate(
        zip(
            production_features,
            generated_features
        )
    ):

        if expected != generated:

            print(
                f"Position {index + 1}"
            )

            print(
                "Expected :",
                expected
            )

            print(
                "Generated:",
                generated
            )

            break


# ============================================================
# 10. DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE FEATURE VALIDATION")
print("=" * 70)

duplicates = (
    feature_vector.columns[
        feature_vector.columns.duplicated()
    ]
    .tolist()
)


print(
    "\nDuplicate generated features:",
    len(duplicates)
)

if duplicates:

    for feature in duplicates:
        print(
            "  -",
            feature
        )

else:

    print(
        "✓ No duplicate features"
    )


# ============================================================
# 11. MISSING VALUE CHECK
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE VALIDATION")
print("=" * 70)

missing_values = (
    feature_vector
    .isna()
    .sum()
)


total_missing = int(
    missing_values.sum()
)


print(
    "\nTotal missing values:",
    total_missing
)


if total_missing > 0:

    print(
        "\nFeatures containing missing values:"
    )

    for feature, count in (
        missing_values[
            missing_values > 0
        ]
        .items()
    ):

        print(
            f"  - {feature}: {count}"
        )

else:

    print(
        "✓ No missing values"
    )


# ============================================================
# 12. FEATURE TYPES
# ============================================================

print("\n" + "=" * 70)
print("FEATURE TYPE INSPECTION")
print("=" * 70)

print(
    "\nGenerated data types:"
)

print(
    feature_vector.dtypes.to_string()
)


# ============================================================
# 13. DISPLAY COMPLETE VECTOR
# ============================================================

print("\n" + "=" * 70)
print("GENERATED 56-FEATURE VECTOR")
print("=" * 70)

for index, feature in enumerate(
    generated_features,
    start=1
):

    value = feature_vector.iloc[
        0
    ][feature]

    print(
        f"{index:02d}. "
        f"{feature:<40} = {value}"
    )


# ============================================================
# 14. FINAL VALIDATION
# ============================================================

schema_valid = (
    feature_vector.shape[0] == 1
    and
    feature_vector.shape[1] == 56
    and
    len(missing_features) == 0
    and
    len(unexpected_features) == 0
    and
    order_matches
    and
    len(duplicates) == 0
    and
    total_missing == 0
)


print("\n" + "=" * 70)
print("STAGE 9C FINAL RESULT")
print("=" * 70)


if schema_valid:

    print(
        "\n✓ PRODUCTION FEATURE MAPPER VALID"
    )

    print(
        "✓ 1 input row"
    )

    print(
        "✓ Exactly 56 features"
    )

    print(
        "✓ No missing features"
    )

    print(
        "✓ No unexpected features"
    )

    print(
        "✓ Exact feature order"
    )

    print(
        "✓ No duplicate features"
    )

    print(
        "✓ No missing values"
    )

    print(
        "\nREADY FOR STAGE 9D"
    )

    print(
        "MODEL PREDICTION USING FEATURE MAPPER"
    )

else:

    print(
        "\n✗ FEATURE MAPPER VALIDATION FAILED"
    )

    print(
        "\nDO NOT CONNECT THIS MAPPER TO FLASK YET."
    )


print("\n" + "=" * 70)
print("STAGE 9C COMPLETED")
print("=" * 70)

STAGE 9C — FEATURE MAPPER VALIDATION

Project root:
D:\newwwwwwww\AiBasedInstagramPrediction

LOADING FEATURE MAPPER

✓ feature_mapper imported successfully

Production schema loaded
Expected features: 56

TEST INPUT

Caption:
Amazing sunset in Sri Lanka! 🌅✨ Such a beautiful evening by the beach.

Hashtags:
#srilanka #travel #sunset #photography #beach

Image:
D:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\img\insta1.jpg

Image exists: True

BUILDING PRODUCTION FEATURE VECTOR

✓ Feature vector generated

VECTOR SHAPE VALIDATION

Rows: 1
Columns: 56

FEATURE NAME VALIDATION

Missing features: 0

Unexpected features: 0

FEATURE ORDER VALIDATION

Exact feature order: True

DUPLICATE FEATURE VALIDATION

Duplicate generated features: 0
✓ No duplicate features

MISSING VALUE VALIDATION

Total missing values: 0
✓ No missing values

FEATURE TYPE INSPECTION

Generated data types:
category                             str
account_type                         str
follower_cou

In [2]:
# ============================================================
# STAGE 9C — FINAL VALIDATION SUMMARY
# ============================================================

print("=" * 70)
print("STAGE 9C — FINAL VALIDATION SUMMARY")
print("=" * 70)

print("\nGenerated shape:")
print(feature_vector.shape)

print("\nExpected feature count:")
print(len(production_features))

print("\nGenerated feature count:")
print(len(generated_features))

print("\nMissing features:")
print(len(missing_features))

if missing_features:
    for x in missing_features:
        print(" -", x)

print("\nUnexpected features:")
print(len(unexpected_features))

if unexpected_features:
    for x in unexpected_features:
        print(" -", x)

print("\nExact feature order:")
print(order_matches)

print("\nDuplicate features:")
print(len(duplicates))

print("\nMissing values:")
print(total_missing)

print("\n" + "=" * 70)

if schema_valid:
    print("✓ STAGE 9C PASSED")
    print("✓ EXACT 56-FEATURE MAPPER READY")
    print("✓ READY FOR STAGE 9D")
else:
    print("✗ STAGE 9C FAILED")
    print("✗ DO NOT CONNECT TO PRODUCTION MODEL YET")

print("=" * 70)

STAGE 9C — FINAL VALIDATION SUMMARY

Generated shape:
(1, 56)

Expected feature count:
56

Generated feature count:
56

Missing features:
0

Unexpected features:
0

Exact feature order:
True

Duplicate features:
0

Missing values:
0

✓ STAGE 9C PASSED
✓ EXACT 56-FEATURE MAPPER READY
✓ READY FOR STAGE 9D
